In [ ]:
import subprocess, os, shutil

REPO_URL = "https://github.com/safety-research/legibility.git"
REPO_DIR = "/workspace/18-4-2026"
EXP_DIR = os.path.join(REPO_DIR, "experiments", "2026", "15-4-2026")

# Clone or pull latest (fetch + reset to ensure we have the newest commit)
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

# Install git-lfs if not available, then pull LFS files
if shutil.which("git-lfs") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=False)
    subprocess.run(["git", "lfs", "install"], check=False)
subprocess.run(["git", "-C", REPO_DIR, "lfs", "pull"], check=False)

# Install dependencies
req_path = os.path.join(EXP_DIR, "requirements.txt")
if os.path.exists(req_path):
    subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)
else:
    print(f"WARNING: {req_path} not found, skipping pip install")

# Set working directory so Path.cwd().parent resolves to experiment root
os.chdir(os.path.join(EXP_DIR, "notebooks"))

# NB7: Reader-Side Activation Analysis (Experiment D)

**CPU notebook** (~15 min). Analyzes R2 activations to understand what happens
inside the reader when it succeeds vs fails at C2 crossfill.

Split R2 activations by C2 outcome and legibility class. Train probe to predict
C2 success. Include foreignness covariate. Report AUROC with and without foreignness.

**Requires:** NB2 outputs (`activations/R2_last_token/`, `activations/R2_cot_boundary/`)

In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import (
    load_activations, load_foreignness_scores, train_binary_probe,
    permutation_test, plot_layer_probe_curve, bootstrap_ci_metric,
    ACTIVATIONS_DIR, PHASE2_RESULTS_DIR,
)

PHASE2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Checkpoint: check if final output already exists
_output_path = PHASE2_RESULTS_DIR / 'reader_analysis_results.json'
if _output_path.exists():
    import json as _json
    with open(_output_path) as _f:
        _saved = _json.load(_f)
    print(f"CACHED: {_output_path} exists.")
    for key in _saved:
        n_layers = len(_saved[key]) if isinstance(_saved[key], dict) else 'N/A'
        print(f"  {key}: {n_layers} layers")
    print("Delete this file and re-run to recompute.")

In [ ]:
# Load R2 activations and metadata
r2_last = load_activations(ACTIVATIONS_DIR / "R2_last_token")
r2_boundary = load_activations(ACTIVATIONS_DIR / "R2_cot_boundary")

with open(ACTIVATIONS_DIR / "R2_last_token" / "metadata.json") as f:
    r2_meta = json.load(f)

r2_labels = np.array(r2_meta['labels'])  # 1=C2 success, 0=C2 failure
r2_sample_meta = r2_meta['sample_metadata']

print(f"R2 activations: {len(r2_labels)} samples, layers={sorted(r2_last.keys())}")
print(f"C2 success rate: {r2_labels.mean():.1%}")

# Check by legibility class
for label in ['ANSWER_LEAKED', 'REASONING_LEGIBLE', 'ILLEGIBLE']:
    mask = np.array([m['label'] == label for m in r2_sample_meta])
    if mask.sum() > 0:
        print(f"  {label}: n={mask.sum()}, C2 success={r2_labels[mask].mean():.1%}")

In [ ]:
# Probe 1: Predict C2 success from last-token activations (all samples)
print("=== Probe: Predict C2 Success (all samples) ===")

c2_success_results = {}
for layer_idx in sorted(r2_last.keys()):
    features = r2_last[layer_idx]
    n_min_class = min(r2_labels.sum(), (1 - r2_labels).sum())
    if n_min_class < 5:
        print(f"  Layer {layer_idx}: skipping (min class={n_min_class})")
        continue
    
    result = train_binary_probe(features, r2_labels, n_splits=5)
    c2_success_results[layer_idx] = result
    print(f"  Layer {layer_idx:3d}: AUROC={result['auroc']:.3f} CI={result['auroc_ci']}")

if c2_success_results:
    fig, ax = plot_layer_probe_curve(
        c2_success_results,
        title='Exp D: C2 Success Probe (R2 Last Token)',
        save_path=str(PHASE2_RESULTS_DIR / 'd_c2_success_probe.png'),
    )
    plt.show()

In [ ]:
# Probe 2: Predict C2 success from CoT boundary activations
print("=== Probe: Predict C2 Success (CoT boundary) ===")

boundary_results = {}
for layer_idx in sorted(r2_boundary.keys()):
    features = r2_boundary[layer_idx]
    n_min_class = min(r2_labels.sum(), (1 - r2_labels).sum())
    if n_min_class < 5:
        continue
    
    result = train_binary_probe(features, r2_labels, n_splits=5)
    boundary_results[layer_idx] = result
    print(f"  Layer {layer_idx:3d}: AUROC={result['auroc']:.3f} CI={result['auroc_ci']}")

In [ ]:
# Probe 3: Predict legibility class from R2 activations
# (excluding ANSWER_LEAKED)
print("=== Probe: Predict Legibility from R2 Activations ===")

leg_mask = np.array([
    m['label'] in ('REASONING_LEGIBLE', 'ILLEGIBLE')
    for m in r2_sample_meta
])
leg_labels = np.array([
    1 if m['label'] == 'REASONING_LEGIBLE' else 0
    for m in r2_sample_meta
])

print(f"Non-leaked samples: {leg_mask.sum()} "
      f"(legible={leg_labels[leg_mask].sum()}, illegible={(1-leg_labels[leg_mask]).sum()})")

if leg_mask.sum() >= 20:
    legibility_from_reader_results = {}
    for layer_idx in sorted(r2_last.keys()):
        features = r2_last[layer_idx][leg_mask]
        labels = leg_labels[leg_mask]
        
        result = train_binary_probe(features, labels, n_splits=5)
        legibility_from_reader_results[layer_idx] = result
        print(f"  Layer {layer_idx:3d}: AUROC={result['auroc']:.3f} CI={result['auroc_ci']}")
else:
    print("Insufficient non-leaked samples for this analysis")
    legibility_from_reader_results = {}

In [ ]:
# Foreignness covariate analysis
foreignness = load_foreignness_scores()

# Get foreignness for each R2 sample
r2_foreignness = []
for m in r2_sample_meta:
    key = (m['sample_id'], m['generator_id'], m['epoch'], 'R2')
    f_score = foreignness.get(key, np.nan)
    r2_foreignness.append(f_score)
r2_foreignness = np.array(r2_foreignness, dtype=float)

valid_f = np.isfinite(r2_foreignness)
print(f"Foreignness available: {valid_f.sum()}/{len(r2_foreignness)}")

if valid_f.sum() >= 20 and c2_success_results:
    # Compare C2 success probe with vs without foreignness
    best_layer = max(c2_success_results, key=lambda k: c2_success_results[k]['auroc'])
    features_base = r2_last[best_layer][valid_f]
    labels_f = r2_labels[valid_f]
    foreignness_f = r2_foreignness[valid_f]
    
    # Without foreignness
    r_no_f = train_binary_probe(features_base, labels_f)
    print(f"Without foreignness: AUROC={r_no_f['auroc']:.3f} CI={r_no_f['auroc_ci']}")
    
    # With foreignness
    features_with_f = np.column_stack([features_base, foreignness_f.reshape(-1, 1)])
    r_with_f = train_binary_probe(features_with_f, labels_f)
    print(f"With foreignness:    AUROC={r_with_f['auroc']:.3f} CI={r_with_f['auroc_ci']}")
    
    # Foreignness only
    r_f_only = train_binary_probe(foreignness_f.reshape(-1, 1), labels_f)
    print(f"Foreignness only:    AUROC={r_f_only['auroc']:.3f} CI={r_f_only['auroc_ci']}")

In [ ]:
# Save results
def clean_results(d):
    return {int(k): {kk: vv for kk, vv in v.items() if kk not in ('probe_model', 'scaler')}
            for k, v in d.items()}

output = {
    'c2_success_probe': clean_results(c2_success_results),
    'boundary_probe': clean_results(boundary_results),
    'legibility_from_reader': clean_results(legibility_from_reader_results),
}
with open(PHASE2_RESULTS_DIR / 'reader_analysis_results.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f"Saved to {PHASE2_RESULTS_DIR / 'reader_analysis_results.json'}")